In [ ]:
import pandas as pd
import numpy as np

from sklearn import utils

import sklearn.tree as skt
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import lightgbm as lgb

import matplotlib.pyplot as plt
plt.style.use('ggplot')


In [ ]:
!mkdir /kaggle_simulations
!mkdir /kaggle_simulations/agent
!mkdir /kaggle_simulations/agent/saved_model

In [ ]:
!pip install pip --upgrade -q
!pip install kaggle-environments --upgrade -q

##  Train models

In [ ]:
%%time
DATA_FILE = '../input/santaepisodedatalebro/SantaEpisodeData_lebroschar.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'

data = pd.read_parquet(DATA_FILE)
# data = pd.read_csv(DATA_FILE)
data = data.fillna(0)

model = skt.DecisionTreeRegressor(min_samples_leaf=40)

# model = xgb.XGBRegressor()

# model = LogisticRegression()

# lab_enc = preprocessing.LabelEncoder()
# training_scores_encoded = lab_enc.fit_transform(data[TARGET_COL])
# print(training_scores_encoded)
# print(utils.multiclass.type_of_target(data[TARGET_COL]))
# print(utils.multiclass.type_of_target(data[TARGET_COL].astype('int')))
# print(utils.multiclass.type_of_target(training_scores_encoded))

model.fit(data[TRAIN_FEATS].values, data[TARGET_COL].values)

In [ ]:
# %%time
# DATA_FILE = '../input/santa2020top15lb-data/SantaEpisodeData_top15LB.parquet'
# TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
# TARGET_COL = 'payout'

# data = pd.read_parquet(DATA_FILE)
# # data = pd.read_csv(DATA_FILE)
# data = data.fillna(0)

# print("Data pulled")

# model1 = skt.DecisionTreeRegressor(min_samples_leaf=40)

# # model = xgb.XGBRegressor()

# # model = LogisticRegression()

# # lab_enc = preprocessing.LabelEncoder()
# # training_scores_encoded = lab_enc.fit_transform(data[TARGET_COL])
# # print(training_scores_encoded)
# # print(utils.multiclass.type_of_target(data[TARGET_COL]))
# # print(utils.multiclass.type_of_target(data[TARGET_COL].astype('int')))
# # print(utils.multiclass.type_of_target(training_scores_encoded))

# model1.fit(data[TRAIN_FEATS].values, data[TARGET_COL].values)

In [ ]:
%%time

DATA_FILE = '../input/santa2020top15lb-data/SantaEpisodeData_top15LB.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'

data = pd.read_parquet(DATA_FILE)
# data = pd.read_csv(DATA_FILE)
data = data.fillna(0)

print("Data pulled")

#Setup a regressor
hyper_params = {
    'learning_rate': 0.05,
    "max_depth": 8,
    "n_estimators": 100,
    "subsample": 0.8,
    "random_state": 101,
#     "verbose": 1
    "verbosity": 3,
    "n_jobs": -1
    
}

#reg = RandomForestRegressor()
# reg = GradientBoostingRegressor(**hyper_params)
reg = xgb.XGBRegressor(**hyper_params)
model1=reg.fit(data[TRAIN_FEATS].values, data[TARGET_COL].values)

# pred = model1.predict(X_test)
# #Cost Function
# mse = mean_squared_error(y_test,pred)
# print (reg.score(X_test,y_test))
# print (mse)

## Save models

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/model.sav'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/model1.sav'
pickle.dump(model1, open(filename, 'wb'))

In [ ]:
!ls /kaggle_simulations/agent/saved_model/

## Define Champion and Challenger

In [ ]:
%%writefile /kaggle_simulations/agent/champion.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/saved_model/model.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index]
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index]
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

In [ ]:
%%writefile /kaggle_simulations/agent/challenger.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/saved_model/model1.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index]
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index]
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

In [ ]:
%%writefile agent.py

import random

def random_agent(observation, configuration):
    return random.randrange(configuration.banditCount)

## Run simulations 

In [ ]:
def print_rounds(file1, file2, N=3):
    env = make("mab", debug=True)
    p1_count=0
    p2_count=0
    print ('simulating...',N,'games')
    for i in range(N):
        game=env.run([file1, file2])
        p1_score = env.steps[-1][0]['reward']
        p2_score = env.steps[-1][1]['reward']
        if p1_score>p2_score:
            p1_count+=1
        else:
            p2_count+=1
        env.reset()
        z=i+1
        print(f"Round {i+1}: {p1_score} - {p2_score}")
#     print (p1_count,'for',z,round(p1_count/z,3),'.vs',round(p2_count/z,3))
    print("Champion wins: {} Challenger wins: {}".format(p1_count, p2_count))
    print("Champion win ratio: {} Challenger win ratio: {}".format(round(p1_count/z,3), round(p2_count/z,3)))
    print ('complete')
    points_est1=[]
    points_est2=[]
    
    for x in range(2000):
        #print (game[x][1]['reward'])
        z=x+1
        points_est1.append(game[x][0]['reward']/z)
        points_est2.append(game[x][1]['reward']/z)
        
    plt.plot(points_est1,label='champion')
    plt.plot(points_est2, label='challenger')
    plt.legend()
    plt.show()
    print (sum(points_est2)/len(points_est2))

In [ ]:
print('champion vs challenger')
print_rounds("/kaggle_simulations/agent/champion.py", "/kaggle_simulations/agent/challenger.py", 20)

In [ ]:
from kaggle_environments import make

env = make("mab", debug=True)

env.run(["/kaggle_simulations/agent/champion.py", "/kaggle_simulations/agent/challenger.py"])
env.render(mode="ipython", width=800, height=800)

## Save final model

In [ ]:
# !mkdir /kaggle_simulations/agent/final_model
!cp /kaggle_simulations/agent/saved_model/model1.sav  /kaggle_simulations/agent/final_model/
!mv /kaggle_simulations/agent/final_model/model1.sav /kaggle_simulations/agent/final_model/model.sav

In [ ]:
%%writefile /kaggle_simulations/agent/main.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/final_model/model.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index]
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index]
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

In [ ]:
!cd /kaggle_simulations/agent && tar -czvf /kaggle/working/submit1.tar.gz main.py final_model